# Proxax Optimization Sandbox — Interactive Colab Notebook

Companion notebook to the interactive **Optimization Sandbox** on [wwkong.github.io](https://wwkong.github.io).

This notebook runs real **JAX** + [**Proxax**](https://proxax.readthedocs.io/en/latest/) (`px.solvers.APD`) on the smooth nonconvex and nonsmooth composite benchmarks featured on the homepage:
1. **2D Rosenbrock** (nonconvex narrow valley)
2. **Chained Rosenbrock 50D** (high-dimensional nonconvex)
3. **Sparse Lasso 64D** (nonsmooth $\ell_1$ composite with exact soft-thresholding)
4. **Rotated Spectrum Quadratic 100D** (dense ill-conditioned convex)

In [ ]:
!pip install -q proxax

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from proxax import px

jax.config.update('jax_enable_x64', True)
print('JAX backend:', jax.default_backend())

## 1. 2D Rosenbrock (Nonconvex Smooth)
$$f(x_1, x_2) = (1 - x_1)^2 + b(x_2 - x_1^2)^2, \quad b = 100$$

In [ ]:
b = 100.0
def rosenbrock(x):
  return (1.0 - x[0]) ** 2 + b * (x[1] - x[0] ** 2) ** 2

sol_rosen = px.solve(
    problem=px.CompositeProblem(smooth=rosenbrock),
    x0=jnp.array([-1.2, 1.0]),
    solver=px.solvers.APD(opt_tol=1e-6),
)
print('Solution x*:', sol_rosen.params)
print('Final f(x*):', float(rosenbrock(sol_rosen.params)))

## 2. Chained Rosenbrock 50D (High-Dim Nonconvex)
$$f(x) = \sum_{i=1}^{49} \bigl[(1 - x_i)^2 + b(x_{i+1} - x_i^2)^2\bigr], \quad x_0 = (-1, 1, -1, 1, \dots)$$

In [ ]:
def chained_rosenbrock(x, b=50.0):
  return jnp.sum((1.0 - x[:-1]) ** 2 + b * (x[1:] - x[:-1] ** 2) ** 2)

x0_chain = jnp.where(jnp.arange(50) % 2 == 0, -1.0, 1.0)
sol_chain = px.solve(
    problem=px.CompositeProblem(smooth=chained_rosenbrock),
    x0=x0_chain,
    solver=px.solvers.APD(opt_tol=1e-6),
)
print('Final f(x):', float(chained_rosenbrock(sol_chain.params)))
print('Max |x_i - 1|:', float(jnp.max(jnp.abs(sol_chain.params - 1.0))))

## 3. Sparse Lasso 64D (Nonsmooth $\ell_1$ Composite)
$$\min_{x \in \mathbb{R}^{64}} \tfrac{1}{2}\|Ax - y\|_2^2 + \lambda \|x\|_1$$
Notice how `px.solvers.APD` applies exact soft-thresholding proximal steps (`px.prox.L1`) to recover exact zeros on off-support coordinates.

In [ ]:
key = jax.random.PRNGKey(1337)
m, d = 48, 64
A = jax.random.normal(key, (m, d)) / jnp.sqrt(m)
x_true = jnp.zeros(d)
for idx, val in [(5, 0.85), (14, -0.70), (23, 0.95), (31, -0.65), (40, 0.80), (49, -0.90), (56, 0.68), (61, -0.75)]:
  x_true = x_true.at[idx].set(val)
y = A @ x_true

lam = 0.15
problem_lasso = px.CompositeProblem(
    smooth=lambda x: 0.5 * jnp.sum((A @ x - y) ** 2),
    proximal=px.prox.L1(weight=lam),
)
sol_lasso = px.solve(
    problem=problem_lasso,
    x0=jnp.zeros(d),
    solver=px.solvers.APD(opt_tol=1e-6),
)
x_hat = sol_lasso.params
print('Nonzero coordinates recovered:', int(jnp.sum(jnp.abs(x_hat) > 1e-8)), '/ 64')

plt.figure(figsize=(9, 3.2))
plt.bar(range(d), jnp.abs(x_true), color='lightgray', label='True |x_i|')
plt.scatter(range(d), jnp.maximum(jnp.abs(x_hat), 1e-8), color='#10b981', s=24, zorder=3, label='Proxax APD |x_i|')
plt.yscale('log')
plt.ylim(1e-8, 1.2)
plt.xlabel('Coordinate i')
plt.ylabel('|x_i| (log scale)')
plt.title('Sparse Lasso 64D — Exact Sparsity Recovery with Proxax APD')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Dense Rotated Spectrum Quadratic 100D
$$f(x) = \tfrac{1}{2}(x - x^*)^\top Q \Lambda Q^\top (x - x^*), \quad \lambda_i \in [1, \kappa]$$

In [ ]:
d_q = 100
kappa = 2000.0
eigs = jnp.geomspace(1.0, kappa, d_q)
Q, _ = jnp.linalg.qr(jax.random.normal(jax.random.PRNGKey(42), (d_q, d_q)))
x_star = 0.5 * jnp.sin(0.25 * jnp.arange(1, d_q + 1)) + jnp.where(jnp.arange(d_q) % 2 == 0, 0.25, -0.25)

def spec_quad(x):
  z = Q.T @ (x - x_star)
  return 0.5 * jnp.sum(eigs * z ** 2)

sol_quad = px.solve(
    problem=px.CompositeProblem(smooth=spec_quad),
    x0=jnp.zeros(d_q),
    solver=px.solvers.APD(opt_tol=1e-6),
)
print('Final f(x):', float(spec_quad(sol_quad.params)))
print('||x - x*||_2:', float(jnp.linalg.norm(sol_quad.params - x_star)))